# Update Annual Arrests in Colorado by Offense 1970-2022

* https://data.colorado.gov/Demographics/Annual-Arrests-in-Colorado-by-Offense-1970-2022/xi5f-mkzt/about_data
* Reads the FBI Crime API to get data for Colorado
* Each police district/precinct requires a separate api call, so > 240 api calls...

## Imports

In [1]:
import pandas as pd
import requests
import json
from pathlib import Path
from zipfile import ZipFile

## API

### Get Agency Codes 

In [2]:
f = open("creds.txt")
api_key=f.readline()
api_key=api_key.strip()

url=f"https://api.usa.gov/crime/fbi/cde/agency/byStateAbbr/CO?API_KEY={api_key}"
response_API = requests.get(url
)
#print(response_API.status_code)
data = response_API.text
parse_json = json.loads(data)


In [4]:
agencies = {}
for agc in parse_json:
    print(agc)
 #   print(agc['ori'],agc["agency_name"],agc["agency_id"],agc["county_name"])
 #   agencies[agc['ori']] = agc["agency_name"]
    
print(agencies)

BACA
BENT
LAKE
MESA
PARK
WELD
YUMA
ADAMS
DELTA
EAGLE
GRAND
KIOWA
LOGAN
OTERO
OURAY
ROUTT
CUSTER
DENVER
ELBERT
GILPIN
MOFFAT
MORGAN
PITKIN
PUEBLO
SUMMIT
TELLER
ALAMOSA
BOULDER
CHAFFEE
CONEJOS
CROWLEY
DOLORES
DOUGLAS
EL PASO
FREMONT
JACKSON
LARIMER
LINCOLN
MINERAL
PROWERS
ARAPAHOE
CHEYENNE
COSTILLA
GARFIELD
GUNNISON
HINSDALE
HUERFANO
LA PLATA
MONTROSE
PHILLIPS
SAGUACHE
SAN JUAN
SEDGWICK
ARCHULETA
JEFFERSON
MONTEZUMA
BROOMFIELD
KIT CARSON
LAS ANIMAS
RIO BLANCO
RIO GRANDE
SAN MIGUEL
WASHINGTON
ADAMS, WELD
CLEAR CREEK
BOULDER, WELD
EAGLE, PITKIN
LARIMER, WELD
NOT SPECIFIED
EL PASO, TELLER
ADAMS, JEFFERSON
ARAPAHOE, JEFFERSON
RIO GRANDE, SAGUACHE
ADAMS, ARAPAHOE, DOUGLAS
ARAPAHOE, DOUGLAS, JEFFERSON
{}


### Get Crimes by Agency

In [4]:
# ori = Orginating Agency Identifier

def decodeCrime(parse_json,ori):
    columns = ["data_year"]
    columns[1:] = [key for key in parse_json["keys"]]
    dd = {}
    for col in columns:
        dd[col] = []
    for val in parse_json["data"]:
#        print(ori,val["data_year"])
        for k,v in val.items():

            dd[k].append(v)
    df = pd.DataFrame(dd)
    df.insert(0,column="Agency",value=agencies[ori])
    df.insert(1,column="ori",value=ori)
    
    return df

nagc=0
for ori in agencies:
    nagc+=1
    print(f"{nagc} of {len(agencies)} -  {ori}")
    url=f"https://api.usa.gov/crime/fbi/cde/arrest/agency/{ori}/all?from=1970&to=2023&API_KEY={api_key}"
    response_API = requests.get(url)
    #print(response_API.status_code)
    data = response_API.text
    parse_json = json.loads(data)
#    print(parse_json)
    tmp = decodeCrime(parse_json,ori)
    if nagc > 1:
        crime = pd.concat([crime,tmp])
    else:
        crime=tmp
 
print("DONE ")

1 of 249 -  CO0010000
2 of 249 -  CO0010100
3 of 249 -  CO0010200
4 of 249 -  CO0010300
5 of 249 -  CO0010400


KeyboardInterrupt: 

In [ ]:
crime.shape

In [ ]:
crime['data_year'].value_counts().sort_index()

In [ ]:
crime.rename(columns={"data_year":"year"},inplace=True)

In [ ]:
columns = crime.columns


In [ ]:
for col in columns[2:]:
    crime[col] = crime[col].astype(int)

In [ ]:
crime.dtypes

In [ ]:
columnMap = {
 'Aggravated Assault':'aggravatedAssault',
 'All Other Offenses (Except Traffic)':'allOtherOffensesExceptTraffic',
 'Arson': 'arson',
 'Burglary':'burglary',
 'Curfew and Loitering Law Violations': 'curfewLoiteringLawViolations',
 'Disorderly Conduct':'disorderlyConduct',
 'Driving Under the Influence':'drivingUnderTheInfluence',
 'Drug Abuse Violations - Grand Total':'drugAbuseViolationsGrandTotal',
 'Drunkenness':'drunkenness',
 'Embezzlement':'embezzlement',
 'Forgery and Counterfeiting':'forgeryAndCounterfeiting',
 'Fraud':'fraud',
 'Gambling - Total':'gamblingTotal',
 'Human Trafficking - Commercial Sex Acts':'humanTraffickingCommercialSexActs',
 'Human Trafficking - Involuntary Servitude':'humanTraffickingInvoluntaryServitude',
 'Larceny - Theft':'larcenyTheft',
 'Liquor Laws':'liquorLaws',
 'Manslaughter by Negligence':'manslaughterByNegligence',
 'Motor Vehicle Theft':'motorVehicleTheft',
 'Murder and Nonnegligent Manslaughter': 'murderAndNonnegligentManslaughter',
 'Offenses Against the Family and Children': 'offensesAgainstTheFamilyAndChildren',
 'Prostitution and Commercialized Vice': 'prostitutionAndCommercializedVice',
 'Rape':'rape',
 'Robbery':'robbery',
 'Sex Offenses (Except Rape, and Prostitution and Commercialized Vice)':'sexOffensesExceptRapeAndProstitutionAndCommercializedVice',
 'Simple Assault':'simpleAssault',
 'Stolen Property: Buying, Receiving, Possessing':'stolenPropertyBuyingReceivingPossessing',
 'Suspicion':'suspicion',
 'Vagrancy':'vagrancy',
 'Vandalism':'vandalism',
 'Weapons: Carrying, Possessing, Etc.':'weaponsCarryingPossessingEtc'
}  

crime.rename(columns=columnMap,inplace=True)

In [ ]:
crime.rename(columns={'Agency':'agency'},inplace=True)

In [ ]:
for col in crime.columns:
    print(col)

In [ ]:
crime.to_csv("crimeCO-1970-2022.csv",index=False)

In [ ]:
crime.shape

In [ ]:
crime.head()

### Check Output File

In [ ]:
df = pd.read_csv("crimeCO-1970-2022.csv")

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:

response_API = requests.get(
'https://api.usa.gov/crime/fbi/cde/nibrs/state/CO/all/offense/count?API_KEY=iuEO68Q3WH3VppVcdQkMIjb6OSnEJuh4aFkU9cK8')
#print(response_API.status_code)
data = response_API.text
parse_json = json.loads(data)

In [ ]:
parse_json

In [ ]:
types = [
"count",
# "age",    
# "sex",    
# "race",    
# "ethnicity",    
# #"relationship",   on available for victim 
"location",    
"linkedoffense",    
"weapons"  
]

offenses = [
"all",    
"violent-crime",    
"aggravated-assault",
"burglary",    
"larceny",    
"motor-vehicle-theft",    
"homicide",    
"rape",    
"robbery",    
"arson",    
"property-crime",    

]

In [ ]:

#url = f'https://api.usa.gov/crime/fbi/cde/nibrs/agency/CO0010000/{offen}/offender/{typ}?from=2016&to=2021&API_KEY==iuEO68Q3WH3VppVcdQkMIjb6OSnEJuh4aFkU9cK8'

In [ ]:
ori = "CO0010000"
dataAll = {}
dataAll[ori] = {}
# dataAll[ori]["data"] = []
# dataAll[ori]["what"] = []
cat = "offense"
fails=[]
for typ in types:
    for offen in offenses:
        try:
            url = f'https://api.usa.gov/crime/fbi/cde/nibrs/agency/{ori}/{offen}/{cat}/{typ}?from=2016&to=2021&API_KEY=iuEO68Q3WH3VppVcdQkMIjb6OSnEJuh4aFkU9cK8'        
          #  print(url)
            response_API = requests.get(url)
            print(ori,typ,offen,response_API)
            string = f"{cat},{typ},{offen}"
            data = response_API.text
            parse_json = json.loads(data)
            # dataAll[ori]["data"].append(parse_json)
            # dataAll[ori]["what"].append(string)
            if string not in dataAll[ori]:
                dataAll[ori][string] = []
            dataAll[ori][string].append(parse_json)
        except:
            print("FAIL ",string)
            fails.append(string)
        
print("DONE")

In [ ]:
dataAll['CO0010000']

In [ ]:
dataAll['CO0010000']['data'][0]

In [ ]:
print(dataAll['CO0010000']['what'][1])
print(type(dataAll['CO0010000']['data'][1]))
for val in dataAll['CO0010000']['data'][1]['title']:
    print(val)

In [ ]:
response_API = requests.get('https://api.usa.gov/crime/fbi/cde/nibrs/agency/CO0010000/all/offender/count?from=2016&to=2021&API_KEY=iuEO68Q3WH3VppVcdQkMIjb6OSnEJuh4aFkU9cK8')
data = response_API.text
parse_json = json.loads(data)
print(parse_json)

In [ ]:
parse_json

In [ ]:
def seeCols(df):
    for col in sorted(df.columns):
        print(col)

## Read Data

In [ ]:
col = "incident_id"
off = pd.read_csv("data/NIBRS_OFFENSE.csv")
icd = pd.read_csv("data/NIBRS_incident.csv")

In [ ]:
off.columns

In [ ]:
print(off.shape)
print(icd.shape)

## Merge Offenses and Incidents

In [ ]:
off = off.merge(icd,on=col,how="left")

In [ ]:
off.shape

In [ ]:
off[col].nunique()

In [ ]:
seeCols(off)

In [ ]:
#off["OFFENSE_TYPE_ID".lower()].value_counts()
len(off["OFFENSE_code".lower()].value_counts())

In [ ]:
#off["OFFENSE_TYPE_ID"].isna().sum()
off["OFFENSE_CODE".lower()].isna().sum()

## Get Name of Offense Committed

In [ ]:
offType = pd.read_csv("data/NIBRS_OFFENSE_TYPE.csv")

In [ ]:
offType.columns

In [ ]:
#off["OFFENSE_TYPE_ID"].nunique()
off["OFFENSE_CODE".lower()].nunique()

In [ ]:
#offType["OFFENSE_TYPE_ID"].nunique()
offType["OFFENSE_CODE".lower()].nunique()

## Merge Offense Name

In [ ]:
off = off.merge(offType,on="OFFENSE_TYPE_ID",how="left")

In [ ]:
off.shape

In [ ]:
seeCols(off)

In [ ]:
off["OFFENSE_NAME"].nunique()

In [ ]:
off["OFFENSE_NAME"].value_counts()

In [ ]:
off['OFFENSE_CODE'].value_counts()

In [ ]:
off["AGENCY_ID"].nunique()

## Agencies

In [ ]:
agc = pd.read_csv("data/agencies.csv")

In [ ]:
agc.head()

In [ ]:
seeCols(agc)

In [ ]:
#agc["NCIC_AGENCY_NAME"].value_counts()
#agc["MSA_NAME"].value_counts()
#agc["PUB_AGENCY_NAME"].value_counts()
#agc["DIVISION_NAME"].value_counts()
#agc["SUBMITTING_AGENCY_NAME"].value_counts()
#agc["SUBMITTING_AGENCY_NAME"].value_counts()
#agc["UCR_AGENCY_NAME"].value_counts()
#agc["SAI"].value_counts()
#agc["AGENCY_TYPE_NAME"].value_counts()
agc["0.1"].value_counts()



In [ ]:
for col in sorted(agc.columns):
    print(col)
    print(agc[col].value_counts())
    print("-------------------------------")

In [ ]:
agc["AGENCY_ID"].nunique()

In [ ]:
dfFinal = off[[]]

## Merge Agency info

In [ ]:
off = off.merge(agc,on="AGENCY_ID",how="left")

In [ ]:
off.shape

## CIM Data

### Crime Offenses by Police District 2001-2016 in Colorado

In [ ]:
offCIM = pd.read_csv("https://data.colorado.gov/api/views/ya69-n6ta/rows.csv?accessType=DOWNLOAD")

In [ ]:
offCIM.columns

In [ ]:
offCIM["subType"].value_counts()

In [ ]:
len(df.columns)

In [ ]:
offCIM["policeDistrict"].value_counts()

In [ ]:
offCIM["subType"].value_counts()

### Crime Arrests by Police District 2001-2016 in Colorado

In [ ]:
offArrCIM = pd.read_csv("https://data.colorado.gov/api/views/2e5i-5hfy/rows.csv?accessType=DOWNLOAD")

In [ ]:
offArrCIM.head()

In [ ]:
for col in offArrCIM.columns:
    print(col)

In [ ]:
offArrCIM["policeDistrict"].value_counts()

In [ ]:
offArrCIM.loc[offArrCIM[["policeDistrict","type"]].str.contains("Adams County")]

In [ ]:
offArrCIM.groupby(["policeDistrict","type"]).count()

In [ ]:
df.head()

In [ ]:
df.head()

In [ ]:
df.head()

In [ ]:
df = pd.read_csv("https://data.colorado.gov/resource/xi5f-mkzt.csv")

In [ ]:
df.columns

In [ ]:
agc[['ncic_agency_name','pub_agency_name','population','county_name']].head()

In [ ]:
agc.columns

In [ ]:
agc
agc[["agency_id","ori",'ncic_agency_name','pub_agency_name','population','county_name',
    'agency_type_name']].to_csv("agency_short.csv",index=False)